
# Pemodelan Risiko Penyakit Kronis — Skrining Sleman 2025

Melatih dan mengevaluasi model prediksi risiko untuk empat penyakit (Diabetes Melitus,
Hipertensi, Kanker Paru, TBC) memakai dataset hasil `Persiapan_Data_Revised.ipynb`.

> ## HASIL FINAL
>
> Notebook ini dijalankan dengan **`QUICK_MODE = False`** (30 kandidat, CV 5 lipat).
> **Angka pada notebook ini sudah final dan layak dilaporkan.**
>
> Ringkasan hasil juga tersimpan di `Data Setiap Target/ringkasan_model.csv`.

---

## Posisi notebook ini dalam kerangka KDD

Tinjauan Pustaka §2.1.1 menempatkan seluruh pekerjaan ini dalam kerangka **Knowledge
Discovery in Databases (KDD)** (Fayyad dkk., 1996). Tiap tahap KDD dikerjakan di notebook
yang berbeda, jadi notebook ini hanya memegang dua tahap terakhir:

| Tahap KDD | Dikerjakan di | Isi |
|---|---|---|
| 1. Seleksi data | `Analisis Data Mentah.ipynb` | struktur, kelengkapan, gerbang eligibilitas, kolom yang dibuang |
| 2. Preprocessing | `Persiapan_Data_Revised.ipynb` §1–5 | standardisasi label, perbaikan tinggi/berat tertukar, batas plausibilitas per usia |
| 3. Transformasi | `Persiapan_Data_Revised.ipynb` §6 + **notebook ini Bagian 3** | rekalkulasi IMT, klasifikasi kategori, standardisasi & one-hot encoding |
| 4. Data mining (pemodelan) | **notebook ini Bagian 4** | pemilihan fitur, penanganan ketimpangan, pelatihan & penyetelan empat model |
| 5. Interpretasi & evaluasi | **notebook ini Bagian 5–8** | metrik, confusion matrix, pohon keputusan, feature importance, SHAP |

## Alur notebook

| Bagian | Isi |
|---|---|
| 1 | **Data Loading** — muat 4 dataset, bagi data latih & uji |
| 2 | **Data Exploration** — keseimbangan kelas, sebaran fitur, hubungan dengan target |
| 3 | **Preprocessing** — pemilihan fitur & pipeline encoding |
| 4 | **Model Training & Hyperparameter Tuning** — penanganan ketimpangan lalu 4 model |
| 5 | **Model Evaluation** — tabel perbandingan per penyakit |
| 6 | **Visualisasi Decision Tree** — melihat aturan yang dipelajari model |
| 7 | **Feature Importance & SHAP** — fitur mana yang paling berpengaruh, dan mengapa |
| 8 | **Kesimpulan** |

## Model yang dibandingkan

| Model | Peran |
|---|---|
| `dummy` | Pembanding dasar — menebak sesuai proporsi kelas, tanpa melihat fitur sama sekali |
| `logistic` | Regresi logistik, model linier yang mudah ditafsirkan |
| `decision_tree` | Pohon keputusan, aturannya bisa digambar dan dibaca langsung |
| `xgboost` | Gradient boosting, biasanya paling akurat |

`dummy` **tidak pernah ikut dipilih sebagai model terbaik** — ia garis dasar pembanding,
dan justru itu gunanya (lihat Bagian 5).

## Catatan penting dari tahap sebelumnya

1. **Tiap dataset mencakup rentang usia berbeda.** Diabetes & Hipertensi hanya usia 15+,
   Kanker Paru 45+, TBC semua usia. Model hanya berlaku pada rentang usianya masing-masing.
2. **Label Diabetes sangat dipengaruhi aturan `IMT > 25`** dari kuesioner skrining. Ini
   terlihat langsung pada visualisasi pohon di Bagian 6.
3. **Kolom `id` teracak** untuk melindungi data asli BPJS, jadi dibuang saat memuat data.



---
# 1. Data Loading

In [ ]:

# ===================== IMPORTS =====================
import os
import sys
from pathlib import Path
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, ConfusionMatrixDisplay,
)
from xgboost import XGBClassifier

# Penanganan ketimpangan kelas (Tinjauan Pustaka 2.1.3).
# Pipeline imblearn dipakai supaya resampling HANYA berjalan pada lipatan latih.
from imblearn.over_sampling import SMOTENC
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline as ImbPipeline

# Interpretabilitas model (Tinjauan Pustaka 2.1.6).
import shap

# Modul bersama + konfigurasi wilayah. Akar proyek dicari dari folder kerja,
# jadi tidak ada satu pun jalur absolut di notebook ini: seluruh folder boleh
# dipindah atau diserahkan ke orang lain tanpa sel ini perlu diubah.
WILAYAH = "Sleman"

AKAR = Path.cwd()
while not (AKAR / "Bersama").exists():
    AKAR = AKAR.parent
sys.path.insert(0, str(AKAR / "Bersama"))

import konfigurasi as cfg      # jalur + cara baca berkas mentah tiap wilayah
import viz_style as vz          # gaya grafik seragam untuk semua notebook

vz.apply_style()
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)

RANDOM_STATE = 42


In [ ]:

# ===================== KONFIGURASI =====================
DATA_DIR = cfg.folder_data(WILAYAH)

# ---------------------------------------------------------------------
# QUICK_MODE = True  -> 8 kandidat, CV 3 lipat, ~3 menit   (pengembangan)
# QUICK_MODE = False -> 30 kandidat, CV 5 lipat, lebih lama (LAPORAN FINAL)
# ---------------------------------------------------------------------
QUICK_MODE = False

N_ITER = 8 if QUICK_MODE else 30
CV_FOLDS = 3 if QUICK_MODE else 5
CV = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

DISEASES = {
    "Diabetes Melitus": ("dataset_diabetes.csv", "Berisiko Diabetes Melitus", "15+"),
    "Hipertensi":       ("dataset_hipertensi.csv", "Berisiko Hipertensi", "15+"),
    "Kanker Paru":      ("dataset_kanker_paru.csv", "Berisiko Kanker Paru", "45+"),
    "TBC":              ("dataset_tbc.csv", "Berisiko TBC", "semua usia"),
}

print("=" * 72)
if QUICK_MODE:
    print("QUICK_MODE = True   ->  n_iter=8, cv=3   (~3 menit)")
    print()
    print("  PERINGATAN: hasil SEMENTARA, belum layak dilaporkan.")
    print("  Setel QUICK_MODE = False lalu jalankan ulang untuk angka final.")
else:
    print("QUICK_MODE = False  ->  n_iter=30, cv=5   (mode laporan final)")
print("=" * 72)

In [ ]:

# ===================== MUAT DATASET =====================
data = {}

for nama, (fname, target, usia_label) in DISEASES.items():
    d = pd.read_csv(os.path.join(DATA_DIR, fname))

    print("=" * 108)
    print(f"{nama}   |   {fname}   |   {d.shape[0]:,} baris x {d.shape[1]} kolom "
          f"|   usia {usia_label}")
    print("=" * 108)
    display(d.head())

    # id adalah kunci sintetis teracak -- tidak membawa informasi, dibuang.
    data[nama] = {
        "X": d.drop(columns=["id", target]),
        "y": d[target],
        "target": target,
        "usia": usia_label,
        "base_rate": float(d[target].mean()),
    }

In [ ]:

# ===================== BAGI DATA LATIH & UJI =====================
# Dilakukan sekali di sini lalu disimpan, sehingga semua model dievaluasi
# pada baris uji yang persis sama.
ringkas = []

for nama, d in data.items():
    X_tr, X_te, y_tr, y_te = train_test_split(
        d["X"], d["y"],
        test_size=0.2,
        stratify=d["y"],        # menjaga proporsi kelas di kedua bagian
        random_state=RANDOM_STATE,
    )
    d.update({"X_train": X_tr, "X_test": X_te, "y_train": y_tr, "y_test": y_te})

    ringkas.append({
        "Dataset": nama,
        "Rentang usia": d["usia"],
        "N total": f"{len(d['X']):,}",
        "N latih": f"{len(X_tr):,}",
        "N uji": f"{len(X_te):,}",
        "Berisiko": f"{int(d['y'].sum()):,}",
        "Prevalensi (%)": round(d["base_rate"] * 100, 2),
        "Kelas positif": "minoritas" if d["base_rate"] < 0.5 else "MAYORITAS",
    })

display(pd.DataFrame(ringkas))


Perhatikan kolom **Rentang usia** dan **Prevalensi**. Keempat dataset bukan populasi yang
sama, dan proporsi kelasnya sangat berbeda — dua hal yang menentukan cara membaca seluruh
hasil di bawah.


---
# 2. Data Exploration

Sebelum memodelkan, perlu dipastikan seperti apa data yang masuk ke model: seberapa timpang
kelasnya, bagaimana sebaran fiturnya, dan seberapa kuat hubungan tiap fitur dengan target.

In [ ]:

# ===================== STRUKTUR & TIPE DATA =====================
contoh = data["TBC"]["X"]
display(pd.DataFrame({
    "Tipe": contoh.dtypes.astype(str),
    "N unik": contoh.nunique(),
    "Contoh nilai": [contoh[c].dropna().unique()[:3] for c in contoh.columns],
    "Missing": contoh.isna().sum(),
}))

print("Kolom kategorik masih berupa teks; encoding dilakukan di dalam pipeline (Bagian 3).")

In [ ]:

# ===================== KESEIMBANGAN KELAS =====================
komposisi = pd.DataFrame({
    n: d["y"].value_counts().sort_index().rename({0: "Tidak Berisiko", 1: "Berisiko"})
    for n, d in data.items()
}).T[["Tidak Berisiko", "Berisiko"]]
persen = (komposisi.div(komposisi.sum(axis=1), axis=0) * 100).round(2)
display(persen)

WARNA_KELAS = [vz.PALETTE["RISK"]["Tidak Berisiko"], vz.PALETTE["RISK"]["Berisiko"]]
label_x = [f"{n}\n(usia {d['usia']})" for n, d in data.items()]

fig, axes = plt.subplots(1, 2, figsize=vz.FIG_1X2)

komposisi.plot(kind="bar", stacked=True, ax=axes[0], color=WARNA_KELAS)
axes[0].set_title("Jumlah Absolut per Kelas")
axes[0].set_ylabel("Jumlah catatan skrining")
axes[0].set_xticklabels(label_x, rotation=0)
vz.thousands(axes[0])

persen.plot(kind="bar", stacked=True, ax=axes[1], color=WARNA_KELAS)
axes[1].set_title("Komposisi Relatif (100%)")
axes[1].set_ylabel("% dari dataset")
axes[1].set_xticklabels(label_x, rotation=0)
axes[1].set_ylim(0, 100)
axes[1].legend(loc="center left", bbox_to_anchor=(1.01, 0.5))

for i, n in enumerate(persen.index):
    bawah = 0
    for lab in ["Tidak Berisiko", "Berisiko"]:
        v = persen.loc[n, lab]
        if v > 5:
            axes[1].text(i, bawah + v / 2, f"{v:.1f}%", ha="center", va="center",
                         fontsize=11, color="white", weight="bold")
        bawah += v

plt.tight_layout()
plt.show()


**Hipertensi** justru didominasi kelas positif (62.4%), sedangkan **TBC** (6.7%) dan
**Kanker Paru** (9.9%) sangat timpang ke arah negatif.

Ketimpangan ini punya konsekuensi langsung: pada TBC, model yang menebak *"tidak ada yang
berisiko"* untuk semua orang akan mendapat **akurasi 93%** tanpa menemukan satu pun kasus.
Karena itu akurasi saja tidak pernah cukup, dan Bagian 4 memakai pembobotan kelas.

In [ ]:

# ===================== SEBARAN FITUR NUMERIK MENURUT KELAS =====================
FITUR_NUM = ["Usia", "IMT_recalculated", "Tinggibadan", "Berat Badan"]

fig, axes = plt.subplots(4, 4, figsize=(22, 18))

for i, (nama, d) in enumerate(data.items()):
    plot_df = d["X"].copy()
    plot_df["_kelas"] = d["y"].map({0: "Tidak Berisiko", 1: "Berisiko"}).values

    for j, fitur in enumerate(FITUR_NUM):
        ax = axes[i, j]
        for kelas in ["Tidak Berisiko", "Berisiko"]:
            ax.hist(plot_df.loc[plot_df["_kelas"] == kelas, fitur], bins=40,
                    alpha=0.55, density=True, label=kelas,
                    color=vz.PALETTE["RISK"][kelas])
        ax.set_title(f"{nama} — {fitur}", fontsize=11)
        ax.set_ylabel("kepadatan")
        if i == 0 and j == 0:
            ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:

# ===================== HUBUNGAN IMT DENGAN TARGET =====================
bins = [0, 18.5, 25, 30, 35, 100]
labels = ["<18.5", "18.5-25", "25-30", "30-35", ">35"]

fig, axes = plt.subplots(2, 2, figsize=vz.FIG_2X2)

for ax, (nama, d) in zip(axes.flat, data.items()):
    band = pd.cut(d["X"]["IMT_recalculated"], bins=bins, labels=labels, right=False)
    prev = d["y"].groupby(band, observed=True).mean() * 100

    ax.bar(prev.index.astype(str), prev.values, color="steelblue")
    ax.axhline(d["base_rate"] * 100, color="crimson", ls="--",
               label=f"prevalensi keseluruhan ({d['base_rate']*100:.1f}%)")
    ax.set_title(f"{nama} (usia {d['usia']})")
    ax.set_xlabel("Kelompok IMT")
    ax.set_ylabel("% Berisiko")
    ax.legend(fontsize=9)
    vz.annotate_bars(ax, fmt="{:.1f}%")

fig.suptitle("Prevalensi Risiko menurut Kelompok IMT", y=1.00)
plt.tight_layout()
plt.show()


Panel **Diabetes** memperlihatkan lompatan tajam tepat di IMT 25, dari belasan persen
menjadi hampir 100% — bentuk **fungsi tangga**, bukan kenaikan bertahap. Kenaikan sedrastis
itu di satu titik potong menandakan label diabetes diturunkan dari **aturan ambang** pada
kuesioner, bukan dari penilaian klinis yang berdiri sendiri. Bukti lengkapnya ada di
`Analisis Data Mentah.ipynb` Bagian 8 dan `Klasifikasi BMI.ipynb` Bagian 7.

Hipertensi menunjukkan kenaikan kuat tetapi bertahap. Kanker Paru praktis datar.

*Catatan untuk panel TBC:* kenaikannya kemungkinan besar **efek usia yang menyamar**, karena
TBC satu-satunya dataset yang mencakup usia 0-106 — anak-anak punya IMT rendah sekaligus
risiko TBC rendah. Jangan ditafsirkan sebagai "IMT tinggi menaikkan risiko TBC".


---
# 3. Preprocessing

## Fitur yang dipakai

Model memakai enam kolom:

| Kolom | Tipe | Perlakuan |
|---|---|---|
| `Jeniskelamin` | kategorik | One-Hot Encoding |
| `segmen` | kategorik | One-Hot Encoding |
| `Usia` | numerik | Standardisasi |
| `Tinggibadan` | numerik | Standardisasi |
| `Berat Badan` | numerik | Standardisasi |
| `IMT_recalculated` | numerik | Standardisasi |

**Kolom `Kategori_BMI_WHO8` dan `Kategori_BMI_WHO4` sengaja tidak dipakai.** Keduanya hanya
hasil pengelompokan dari `IMT_recalculated` yang sudah ada, sehingga tidak menambah
informasi baru. Uji ablasi pada data Bantul menunjukkan performanya praktis sama persis
(selisih AP 0.0001 pada diabetes); pilihan fitur itu diwarisi apa adanya di sini
supaya kedua wilayah memakai kumpulan fitur yang sama dan bisa dibandingkan. Mengeluarkannya membuat grafik *feature importance* di Bagian 7 jauh
lebih mudah dibaca — kalau IMT hadir dalam dua bentuk sekaligus, model membagi "kredit"
secara sembarang di antara keduanya.

## Kenapa encoding diletakkan di dalam pipeline

`StandardScaler` dan `OneHotEncoder` **belajar** dari data (rata-rata, simpangan baku,
daftar kategori). Kalau dijalankan sebelum pembagian data, informasi dari data uji ikut
bocor ke proses pelatihan. Dengan memasukkannya ke dalam `Pipeline`, keduanya hanya
mempelajari data latih di setiap lipatan cross-validation.

In [ ]:

# ===================== DEFINISI FITUR =====================
NUMERIC_COLS = ["Usia", "Tinggibadan", "Berat Badan", "IMT_recalculated"]
CATEG_COLS = ["Jeniskelamin", "segmen"]
FEATURES = CATEG_COLS + NUMERIC_COLS

# SMOTENC perlu tahu kolom mana yang kategorik, sebagai posisi di dalam FEATURES.
IDX_KATEGORIK = [FEATURES.index(c) for c in CATEG_COLS]

print(f"Fitur kategorik : {CATEG_COLS}   (posisi {IDX_KATEGORIK} di dalam FEATURES)")
print(f"Fitur numerik   : {NUMERIC_COLS}")
print(f"Total           : {len(FEATURES)} kolom")


In [ ]:

# ===================== PIPELINE PREPROCESSING =====================
def build_preprocessor():
    return ColumnTransformer([
        ("num", StandardScaler(), NUMERIC_COLS),
        ("cat", OneHotEncoder(handle_unknown="ignore", drop="first",
                              sparse_output=False), CATEG_COLS),
    ])


# Contoh: kolom apa yang sebenarnya dilihat model setelah encoding?
contoh_pre = build_preprocessor().fit(data["TBC"]["X_train"][FEATURES])
NAMA_FITUR = list(contoh_pre.get_feature_names_out())

print(f"{len(FEATURES)} kolom asli  ->  {len(NAMA_FITUR)} kolom setelah encoding\n")
for f in NAMA_FITUR:
    print("  ", f)


`drop="first"` membuang satu kategori sebagai acuan. Untuk `Jeniskelamin` yang hanya punya
dua nilai, cukup satu kolom (`P` = 1 berarti perempuan, 0 berarti laki-laki). Ini mencegah
kolom yang saling bergantung sempurna, yang bisa mengganggu regresi logistik.


## Pemilihan fitur

Tinjauan Pustaka §2.1.2 menyebut tiga pendekatan pemilihan fitur (Guyon & Elisseeff, 2003).
Berikut posisi masing-masing dalam penelitian ini:

| Pendekatan | Dipakai? | Bentuknya di sini |
|---|---|---|
| **Filter** | Ya | Kekuatan hubungan tiap fitur dengan tiap target diukur lebih dulu, lepas dari algoritma apa pun: korelasi *point-biserial* untuk fitur numerik dan *Cramér's V* untuk fitur kategorik. Dihitung ulang di sel berikut; analisis lengkapnya ada di `Analisis Hubungan Antar Variabel.ipynb` |
| **Embedded** | Ya | `feature_importances_` XGBoost (Bagian 7) dan nilai SHAP (Bagian 7b) — keduanya lahir dari proses pelatihan model itu sendiri, lalu dipakai menilai kembali kegunaan tiap fitur |
| **Wrapper** | Tidak | Wrapper mencari subset fitur terbaik secara iteratif dengan melatih ulang model berkali-kali. Dengan hanya **6 fitur**, seluruh ruang pencariannya cuma 63 subset, dan tidak ada fitur yang mahal dikumpulkan atau jelas-jelas redundan. Biaya komputasinya tidak sebanding dengan manfaatnya |

Hasil filter di bawah dipakai untuk satu keputusan konkret: **`Kategori_BMI_WHO8` dan
`Kategori_BMI_WHO4` dikeluarkan**, karena keduanya hanya pengelompokan ulang dari
`IMT_recalculated` yang sudah ada (lihat tabel fitur di atas).

> **Perhatian membaca tabel.** Angka terbesar di tabel ini — IMT ↔ Diabetes — justru yang
> **paling tidak informatif**, karena label Diabetes memang diturunkan dari ambang IMT
> (Bagian 2). Filter mengukur kekuatan hubungan, bukan apakah hubungan itu bermakna.


In [ ]:

# ===================== FILTER METHOD: KEKUATAN HUBUNGAN TIAP FITUR =====================
# Numerik  -> korelasi point-biserial (bertanda: arah hubungan ikut terbaca)
# Kategorik -> Cramér's V (selalu positif: hanya kekuatan, tanpa arah)
def cramers_v(x, y):
    tab = pd.crosstab(x, y)
    chi2 = stats.chi2_contingency(tab)[0]
    n = tab.to_numpy().sum()
    r, k = tab.shape
    return float(np.sqrt((chi2 / n) / min(r - 1, k - 1)))


baris = {}
for nama, d in data.items():
    kol = {c: float(stats.pointbiserialr(d["y"], d["X"][c]).statistic) for c in NUMERIC_COLS}
    kol.update({c: cramers_v(d["X"][c], d["y"]) for c in CATEG_COLS})
    baris[nama] = kol

filter_df = pd.DataFrame(baris).T[NUMERIC_COLS + CATEG_COLS]
display(filter_df.round(3))

print("Fitur terkuat per penyakit (nilai mutlak):")
for nama in filter_df.index:
    s = filter_df.loc[nama]
    print(f"   {nama:20} -> {s.abs().idxmax():18} ({s[s.abs().idxmax()]:+.3f})")



---
# 4. Model Training & Hyperparameter Tuning

## Menangani ketimpangan kelas

Pada TBC hanya 6.7% peserta berlabel berisiko. Tanpa penanganan, model akan belajar bahwa
menebak "tidak berisiko" untuk semua orang sudah benar 93% — dan recall-nya menjadi nol.

Tinjauan Pustaka §2.1.3 menyebut tiga pendekatan. Ketiganya tersedia di notebook ini dan
**dibandingkan langsung dengan angka** di Bagian 4a:

| Pendekatan | Implementasi | Cara kerjanya |
|---|---|---|
| *(pembanding)* tanpa penanganan | `penyeimbang=None` | tidak ada perlakuan apa pun |
| **Class weight** | `class_weight="balanced"` (Logistic dan Decision Tree) dan `scale_pos_weight` (XGBoost) | kesalahan pada kelas minoritas dihitung lebih mahal |
| **Oversampling — SMOTE** | `SMOTENC` (Chawla dkk., 2002) | menambah baris sintetis untuk kelas minoritas |
| **Undersampling** | `RandomUnderSampler` (He & Garcia, 2009) | membuang sebagian baris kelas mayoritas |

### Kenapa `SMOTENC`, bukan `SMOTE` biasa

SMOTE membuat sampel baru dengan **interpolasi** antara dua sampel minoritas yang
bertetangga. Interpolasi hanya bermakna untuk angka. Kalau dijalankan pada kolom hasil
one-hot, ia akan menghasilkan nilai seperti `segmen_PBPU = 0.37` — peserta yang "37 persen
PBPU", yang tidak ada artinya.

`SMOTENC` adalah varian SMOTE untuk data campuran numerik–kategorik: kolom numerik
diinterpolasi seperti biasa, sedangkan kolom kategorik diisi dengan kategori **terbanyak di
antara tetangga terdekatnya**. Karena itu `SMOTENC` diletakkan **sebelum**
`ColumnTransformer`, yaitu saat `Jeniskelamin` dan `segmen` masih berupa teks asli.

### Letak resampling di dalam pipeline menentukan sah atau tidaknya hasil

Kalau oversampling dijalankan sekali di awal pada seluruh data latih, baris sintetis yang
dibuat dari satu baris bisa jatuh ke lipatan latih sementara "induknya" jatuh ke lipatan
validasi. Model lalu dinilai pada data yang secara tidak langsung sudah pernah dilihatnya,
dan skornya menjadi terlalu bagus — bocor.

Dengan menaruh `SMOTENC` sebagai langkah pertama `imblearn.pipeline.Pipeline`, resampling
hanya berjalan pada **lipatan latih di setiap iterasi cross-validation**, dan tidak pernah
berjalan saat `predict`. **Data uji selalu memakai distribusi aslinya**, sehingga metrik di
Bagian 5 tetap menggambarkan populasi yang sebenarnya.

### Pembobotan tidak selalu membantu

Pada **Hipertensi**, kelas positif justru **mayoritas** (62.4%) — yang langka di sana adalah
kelas negatif. Menyeimbangkan kelas di situ berarti mendorong model menjauhi kelas yang
sebenarnya lebih sering benar, dan hasilnya **turun**:

| Hipertensi, XGBoost (diukur pada data **Bantul**) | Accuracy | F1-Score |
|---|---|---|
| dengan pembobotan | 0.665 | 0.655 |
| **tanpa pembobotan** | **0.689** | **0.745** |

*Percobaan itu dijalankan pada data Bantul dan tidak diulang di sini.* Yang membuatnya tetap berlaku untuk Sleman bukan angkanya, melainkan sebabnya: kelas positif Hipertensi juga **mayoritas** di Sleman (62,4%), sehingga pembobotan menghadapi situasi yang sama. Kalau angka khusus Sleman dibutuhkan, percobaannya harus dijalankan ulang di sini.

Karena itu **Hipertensi tidak diikutkan** dalam perbandingan Bagian 4a, dan aturan bawaannya
tetap satu baris: `pakai_bobot = prevalensi < 0.5`.

## Kenapa `scoring="roc_auc"` dan bukan `"f1"`

`RandomizedSearchCV` butuh satu angka untuk membandingkan kandidat hyperparameter.

Kalau memakai `f1`, pada data setimpang hampir semua kandidat menghasilkan F1 = 0.000 yang
sama persis — sehingga pencarian tidak punya dasar untuk membedakan mereka dan akhirnya
memilih secara acak. **ROC-AUC menilai urutan peringkat**, bukan satu titik potong, sehingga
selalu bisa membedakan kandidat. Metrik ini juga sudah masuk tabel evaluasi di Bagian 5.

> Sifat yang sama juga punya sisi buruk, dan akan terlihat jelas di Bagian 4a: karena
> ROC-AUC tidak bergantung pada ambang, ia **hampir tidak bereaksi** terhadap penyeimbangan
> kelas. Jadi ROC-AUC tepat untuk memilih *hyperparameter*, tetapi **tidak** tepat untuk
> memilih *penanganan ketimpangan*.


In [ ]:

# ===================== PEMBANGUN MODEL & RUANG PENCARIAN =====================
# penyeimbang: cara menangani ketimpangan kelas (Tinjauan Pustaka 2.1.3)
#   None           -> tanpa penanganan
#   "class_weight" -> pembobotan kelas
#   "smote"        -> oversampling sintetis dengan SMOTENC
#   "undersample"  -> membuang sebagian baris kelas mayoritas
NAMA_OPSI = {
    None: "tanpa penanganan",
    "class_weight": "class weight",
    "smote": "SMOTE (SMOTENC)",
    "undersample": "undersampling",
}


def build_estimator(model_key, penyeimbang="class_weight", pos_weight=1.0, param_tetap=None):
    pre = build_preprocessor()

    # Pembobotan hanya aktif bila penyeimbang yang dipilih memang class weight.
    pakai_bobot = penyeimbang == "class_weight"
    bobot = "balanced" if pakai_bobot else None
    spw = pos_weight if pakai_bobot else 1.0

    if model_key == "dummy":
        # Pembanding dasar: menebak sesuai proporsi kelas, mengabaikan semua fitur.
        # Tidak menerima penyeimbang apa pun -- justru itu maksudnya.
        return Pipeline([("pre", pre), ("clf", DummyClassifier(strategy="prior"))]), {}

    if model_key == "logistic":
        clf = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE,
                                 class_weight=bobot)
        grid = {"clf__C": [0.001, 0.01, 0.1, 1, 10, 100],
                "clf__solver": ["lbfgs", "liblinear"]}

    elif model_key == "decision_tree":
        clf = DecisionTreeClassifier(random_state=RANDOM_STATE, class_weight=bobot)
        grid = {"clf__criterion": ["gini", "entropy"],
                "clf__max_depth": [3, 5, 7, 10, 15, None],
                "clf__min_samples_split": [2, 10, 50],
                "clf__min_samples_leaf": [1, 5, 20]}

    elif model_key == "xgboost":
        clf = XGBClassifier(objective="binary:logistic", eval_metric="logloss",
                            tree_method="hist", random_state=RANDOM_STATE, n_jobs=1,
                            scale_pos_weight=spw)
        grid = {"clf__n_estimators": [200, 300, 500],
                "clf__max_depth": [3, 4, 6, 8],
                "clf__learning_rate": [0.03, 0.05, 0.1],
                "clf__subsample": [0.8, 1.0],
                "clf__colsample_bytree": [0.8, 1.0],
                "clf__min_child_weight": [1, 3, 5]}
    else:
        raise ValueError(f"model_key tidak dikenal: {model_key}")

    # Hyperparameter dipatok -> tidak ada pencarian (dipakai di Bagian 4a).
    if param_tetap:
        clf.set_params(**{k.replace("clf__", ""): v for k, v in param_tetap.items()})
        grid = {}

    # Resampler SELALU menjadi langkah pertama, sebelum ColumnTransformer:
    #  - SMOTENC butuh kolom kategorik yang masih berupa teks asli
    #  - langkah sampler otomatis dilewati saat predict, jadi data uji tidak tersentuh
    langkah = []
    if penyeimbang == "smote":
        langkah.append(("smote", SMOTENC(categorical_features=IDX_KATEGORIK,
                                         random_state=RANDOM_STATE)))
    elif penyeimbang == "undersample":
        langkah.append(("under", RandomUnderSampler(random_state=RANDOM_STATE)))
    langkah += [("pre", pre), ("clf", clf)]

    Pipe = ImbPipeline if penyeimbang in ("smote", "undersample") else Pipeline
    return Pipe(langkah), grid


def penyeimbang_bawaan(base_rate):
    # Aturan bawaan: seimbangkan hanya bila kelas positif memang minoritas.
    return "class_weight" if base_rate < 0.5 else None


def latih_model(nama, model_key, penyeimbang="bawaan", param_tetap=None):
    # Melatih satu model untuk satu penyakit.
    # Mengembalikan (model terlatih, probabilitas data uji, hyperparameter terpilih).
    d = data[nama]
    X_tr, X_te = d["X_train"][FEATURES], d["X_test"][FEATURES]
    y_tr = d["y_train"]

    if penyeimbang == "bawaan":
        penyeimbang = penyeimbang_bawaan(d["base_rate"])

    # scale_pos_weight = jumlah negatif / jumlah positif pada data latih
    pos_w = float((y_tr == 0).sum() / max((y_tr == 1).sum(), 1))
    pipe, grid = build_estimator(model_key, penyeimbang, pos_w, param_tetap)

    if grid:
        search = RandomizedSearchCV(pipe, grid, n_iter=N_ITER, scoring="roc_auc",
                                    cv=CV, random_state=RANDOM_STATE, n_jobs=-1)
        search.fit(X_tr, y_tr)
        model, best_params = search.best_estimator_, search.best_params_
    else:
        model, best_params = pipe.fit(X_tr, y_tr), (param_tetap or {})

    return model, model.predict_proba(X_te)[:, 1], best_params



## 4a. Perbandingan penanganan ketimpangan

Perbandingan dijalankan hanya pada dua target yang benar-benar timpang ke arah negatif —
**Kanker Paru** (9.9% positif) dan **TBC** (6.7% positif). Diabetes (41.7%) hanya sedikit
timpang dan Hipertensi kelas positifnya mayoritas, jadi keduanya memakai aturan bawaan.

Supaya yang dibandingkan benar-benar *penanganan ketimpangannya*, semua hal lain dibuat
sama: satu algoritma (XGBoost), **satu set hyperparameter tetap**, satu pembagian data.
Hyperparameter sengaja tidak disetel ulang per pendekatan — penyetelan penuh baru dilakukan
sesudah pendekatan terpilih, di Bagian 4.

### Aturan pemilihan — ditetapkan sebelum melihat hasilnya

1. **Buang pendekatan dengan recall < 0.05.** Recall serendah itu berarti model praktis
   tidak menandai siapa pun sebagai berisiko: ia hanya menyalin tebakan kelas mayoritas,
   persis kegagalan yang membuat bagian ini ada.
2. Kalau selisih F1 antar sisanya **di bawah 0.02**, ketiganya dianggap **setara** dan
   dipilih **class weight**.
3. Kalau tidak setara, pilih F1 tertinggi.

Alasan class weight jadi pilihan saat hasilnya setara ada tiga, dan tidak satu pun soal
angka: ia **tidak membuang data asli** (undersampling membuang sebagian besar baris
negatif), **tidak mengarang pasien sintetis** — hal yang canggung dipertanggungjawabkan
pada data kesehatan — dan **paling murah** secara komputasi.


In [ ]:

# ===================== 4a. BANDINGKAN EMPAT PENDEKATAN =====================
# Hyperparameter dipatok supaya satu-satunya yang berubah adalah penanganan ketimpangan.
PROBE = {"n_estimators": 300, "max_depth": 4, "learning_rate": 0.05}
TARGET_TIMPANG = ["Kanker Paru", "TBC"]
OPSI = [None, "class_weight", "smote", "undersample"]

hasil_seimbang, PENYEIMBANG_DIPILIH = [], {}

for nama, d in data.items():
    if nama not in TARGET_TIMPANG:
        PENYEIMBANG_DIPILIH[nama] = penyeimbang_bawaan(d["base_rate"])
        continue

    skor = {}
    for ps in OPSI:
        t = time.time()
        _, p, _ = latih_model(nama, "xgboost", penyeimbang=ps, param_tetap=PROBE)
        y_te = d["y_test"]
        pred = (p >= 0.5).astype(int)
        skor[ps] = {
            "Accuracy": accuracy_score(y_te, pred),
            "Precision": precision_score(y_te, pred, zero_division=0),
            "Recall": recall_score(y_te, pred, zero_division=0),
            "F1-Score": f1_score(y_te, pred, zero_division=0),
            "ROC-AUC": roc_auc_score(y_te, p),
        }
        hasil_seimbang.append({"Dataset": nama, "Penanganan": NAMA_OPSI[ps],
                               **skor[ps], "Detik": round(time.time() - t, 1)})

    # --- aturan pemilihan (lihat markdown di atas) ---
    layak = {k: v for k, v in skor.items() if v["Recall"] >= 0.05}
    f1 = {k: v["F1-Score"] for k, v in layak.items()}
    if max(f1.values()) - min(f1.values()) < 0.02:
        PENYEIMBANG_DIPILIH[nama] = "class_weight"
        alasan = f"selisih F1 hanya {max(f1.values()) - min(f1.values()):.4f} (< 0.02) -> setara"
    else:
        PENYEIMBANG_DIPILIH[nama] = max(f1, key=f1.get)
        alasan = f"F1 tertinggi ({max(f1.values()):.4f})"
    print(f"{nama:14} -> {NAMA_OPSI[PENYEIMBANG_DIPILIH[nama]]:18} | {alasan}")

display(pd.DataFrame(hasil_seimbang).set_index(["Dataset", "Penanganan"]).round(4))

print("\nPendekatan yang dipakai di Bagian 4:")
for nama, ps in PENYEIMBANG_DIPILIH.items():
    catatan = "" if nama in TARGET_TIMPANG else "   (aturan bawaan, tidak diuji)"
    print(f"   {nama:20} -> {NAMA_OPSI[ps]}{catatan}")



### Membaca tabel di atas

Tiga hal yang layak dibawa ke laporan:

**1. Tanpa penanganan, modelnya tidak berguna — meski akurasinya paling tinggi.** Baris
`tanpa penanganan` mencetak akurasi tertinggi di kedua penyakit sekaligus recall yang
praktis nol. Ia menandai hampir tidak ada orang sebagai berisiko. Untuk alat skrining, itu
kegagalan total, bukan keberhasilan.

**2. ROC-AUC hampir tidak bergerak.** Selisihnya antar keempat pendekatan sangat kecil,
padahal recall-nya berbeda jauh. Sebabnya struktural: penyeimbangan kelas tidak mengubah
**urutan** peserta dari yang paling berisiko ke yang paling tidak — ia hanya menggeser
letak ambang 0.5 pada urutan itu. ROC-AUC menilai urutan, jadi ia memang buta terhadap
perubahan ini. **ROC-AUC tepat untuk memilih hyperparameter, tidak tepat untuk memilih
penanganan ketimpangan.**

**3. Ketiga penanganan sampai ke tempat yang sama.** Selisih F1 di antara class weight,
SMOTE, dan undersampling ada di bawah 0.01 pada kedua penyakit. Tidak ada pendekatan yang
"menang" dalam arti berarti — yang penting adalah memakai salah satunya.

Karena setara, pilihannya jatuh ke class weight dengan alasan yang sudah disebut di atas.
Perlu dicatat bahwa SMOTE menaikkan precision tetapi **menurunkan recall cukup jauh** pada
TBC; untuk alat skrining, menukar kasus yang tertangkap demi alarm palsu yang lebih sedikit
adalah arah yang keliru.


In [ ]:

# ===================== LATIH SEMUA MODEL =====================
MODEL_KEYS = ["dummy", "logistic", "decision_tree", "xgboost"]

models, probs, params = {}, {}, {}
t0 = time.time()

for nama, d in data.items():
    ps = PENYEIMBANG_DIPILIH[nama]
    print(f"\n{nama}   |   prevalensi {d['base_rate']*100:.1f}%   "
          f"|   penanganan ketimpangan: {NAMA_OPSI[ps]}")
    for mk in MODEL_KEYS:
        t = time.time()
        mdl, p, bp = latih_model(nama, mk, penyeimbang=ps)
        models[(nama, mk)], probs[(nama, mk)], params[(nama, mk)] = mdl, p, bp
        print(f"   {mk:<15} selesai dalam {time.time()-t:>6.1f} detik")

print(f"\nTotal waktu pelatihan: {(time.time()-t0)/60:.1f} menit")


In [ ]:

# ===================== HYPERPARAMETER TERPILIH =====================
baris = []
for (nama, mk), bp in params.items():
    if bp:
        baris.append({"Dataset": nama, "Model": mk,
                      **{k.replace("clf__", ""): v for k, v in bp.items()}})

display(pd.DataFrame(baris).set_index(["Dataset", "Model"]).sort_index())


---
# 5. Model Evaluation

Lima metrik dilaporkan untuk tiap model:

| Metrik | Arti | Catatan |
|---|---|---|
| **Accuracy** | Proporsi prediksi yang benar | Menyesatkan pada data timpang — lihat baris `dummy` |
| **Precision** | Dari yang ditandai berisiko, berapa yang benar | Tinggi = sedikit "alarm palsu" |
| **Recall** | Dari yang benar-benar berisiko, berapa yang tertangkap | Tinggi = sedikit kasus terlewat |
| **F1-Score** | Rata-rata harmonik Precision & Recall | Ringkasan keseimbangan keduanya |
| **ROC-AUC** | Kemampuan mengurutkan risiko | 0.5 = menebak acak, 1.0 = sempurna |

Kolom **Prevalensi** disertakan sebagai konteks, bukan sebagai metrik — tanpa itu angka
akurasi tidak bisa ditafsirkan.

In [ ]:

# ===================== FUNGSI EVALUASI =====================
def evaluasi(nama, model_key):
    y_te = data[nama]["y_test"]
    prob = probs[(nama, model_key)]
    pred = (prob >= 0.5).astype(int)

    return {
        "Model": model_key,
        "Accuracy": accuracy_score(y_te, pred),
        "Precision": precision_score(y_te, pred, zero_division=0),
        "Recall": recall_score(y_te, pred, zero_division=0),
        "F1-Score": f1_score(y_te, pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_te, prob),
    }


def model_terbaik(tab):
    # Satu definisi "model terbaik", dipakai di Bagian 5, confusion matrix, dan Bagian 8,
    # supaya ketiganya tidak mungkin menunjuk model yang berbeda.
    #
    # `dummy` DIKECUALIKAN, dan ini bukan sekadar kerapian: pada Hipertensi kelas
    # positif adalah mayoritas, sehingga dummy mencetak F1 0,7552 -- lebih tinggi
    # daripada model sungguhan mana pun. Tanpa pengecualian ini, idxmax memilih
    # dummy dan grafik confusion matrix menampilkan dummy dengan judul
    # "Model Terbaik", bertentangan dengan tabel tepat di atasnya.
    kandidat = tab[tab["Model"] != "dummy"]
    return kandidat.loc[kandidat["F1-Score"].idxmax(), "Model"]


In [ ]:

# ===================== TABEL PERBANDINGAN PER PENYAKIT =====================
tabel_per_penyakit = {}

for nama, d in data.items():
    tab = pd.DataFrame([evaluasi(nama, mk) for mk in MODEL_KEYS])
    tab.insert(1, "Prevalensi", d["base_rate"])
    tabel_per_penyakit[nama] = tab

    print("=" * 92)
    print(f"{nama}   |   usia {d['usia']}   |   n uji = {len(d['y_test']):,}   "
          f"|   prevalensi = {d['base_rate']*100:.2f}%")
    print("=" * 92)
    display(tab.set_index("Model").round(4))
    print(f"Model terbaik menurut F1-Score: {model_terbaik(tab)}\n")



### Cara membaca tabel di atas

Bandingkan tiap model dengan baris **`dummy`** — model yang sengaja tidak belajar apa pun.

Pada **TBC**, `dummy` mencetak **akurasi 93.3%** dengan **F1 = 0.000**. Ia menebak
"tidak berisiko" untuk semua orang dan tidak menemukan satu pun kasus. Inilah alasan
akurasi tidak boleh dilaporkan sendirian, dan alasan Bagian 4a menolak pendekatan "tanpa
penanganan" meski akurasinya paling tinggi.

Pada **Hipertensi**, `dummy` justru sebaliknya: karena kelas positif adalah mayoritas
(62.4%), ia menebak "berisiko" untuk semua orang sehingga Recall = 1.00 dan F1 mencapai
**0.768** — bahkan **lebih tinggi daripada F1 model sungguhan**.

Ini bukan berarti modelnya gagal. F1 hanya memperhitungkan kelas positif dan mengabaikan
seluruh *true negative*, sehingga strategi "tebak positif untuk semua orang" mendapat nilai
tinggi secara artifisial ketika kelas positif memang mayoritas. Lihat metrik lain pada baris
yang sama: `dummy` kalah pada **Accuracy** dan mendapat **ROC-AUC 0.500** — ia sama
sekali tidak bisa membedakan siapa yang lebih berisiko.

Karena itu `dummy` **dikecualikan** dari pemilihan model terbaik: ia garis dasar
pembanding, bukan kandidat. Pengecualian itu bukan formalitas — pada Hipertensi, tanpa
pengecualian tersebut `dummy` akan terpilih sebagai "model terbaik" dan muncul pada grafik
confusion matrix di bawah.

**ROC-AUC adalah metrik yang paling jujur di sini.** `dummy` selalu mendapat **0.500**
persis di keempat penyakit, apa pun proporsi kelasnya. Kalau harus memilih satu metrik
untuk membandingkan *model*, gunakan ROC-AUC — dan gunakan F1 hanya sesudah memastikan ia
tidak sedang dipermainkan oleh proporsi kelas seperti pada Hipertensi.

> Perhatikan bahwa saran ini kebalikan dari Bagian 4a. Untuk membandingkan **model** pada
> data yang sama, ROC-AUC paling jujur. Untuk membandingkan **penanganan ketimpangan**,
> ROC-AUC justru buta. Metrik yang tepat tergantung apa yang sedang dibandingkan.


In [ ]:

# ===================== CONFUSION MATRIX (MODEL TERBAIK) =====================
fig, axes = plt.subplots(2, 2, figsize=vz.FIG_2X2)

for ax, (nama, d) in zip(axes.flat, data.items()):
    mk = model_terbaik(tabel_per_penyakit[nama])

    pred = (probs[(nama, mk)] >= 0.5).astype(int)
    cm = confusion_matrix(d["y_test"], pred, normalize="true")

    ConfusionMatrixDisplay(cm, display_labels=["Tidak Berisiko", "Berisiko"]).plot(
        ax=ax, cmap="Blues", colorbar=False, values_format=".3f")
    ax.set_title(f"{nama} — {mk}\n(usia {d['usia']})")

fig.suptitle("Confusion Matrix Model Terbaik — dinormalisasi per baris", y=1.00)
plt.tight_layout()
plt.show()



Karena dinormalisasi per baris, angka pada **diagonal** adalah proporsi tiap kelas yang
berhasil ditebak benar. Sel kanan-atas = kasus berisiko yang **terlewat**; sel kiri-bawah =
orang sehat yang **ditandai keliru**.


---
# 6. Visualisasi Decision Tree

Keunggulan utama Decision Tree adalah aturannya bisa **dilihat langsung**, tidak seperti
XGBoost yang berisi ratusan pohon.

> **Penting untuk presentasi.** Pohon yang digambar di bawah dibatasi `max_depth=3` agar
> muat dan terbaca. Model Decision Tree hasil tuning di Bagian 4 kemungkinan **lebih dalam**
> (lihat tabel hyperparameter). Jadi gambar ini adalah **versi sederhana untuk interpretasi**,
> bukan model yang dipakai menghasilkan angka di Bagian 5. Sebutkan perbedaan ini bila
> ditanya.

Cara membaca tiap kotak:

- **baris pertama** — aturan pemisah, contoh `num__IMT_recalculated <= 0.35`
- **`samples`** — banyaknya data latih yang masuk ke simpul itu
- **`value`** — sebaran kelas `[Tidak Berisiko, Berisiko]`
- **`class`** — kelas mayoritas di simpul itu
- **warna** — makin pekat berarti simpul makin murni (didominasi satu kelas)

Nilai ambang tampil dalam **skala terstandardisasi** karena `StandardScaler` ada di dalam
pipeline. Yang penting bukan angka mentahnya, melainkan **fitur mana yang dipakai memisah**.

In [ ]:

# ===================== POHON KEPUTUSAN PER PENYAKIT =====================
pohon_sederhana = {}

for nama, d in data.items():
    pohon = Pipeline([
        ("pre", build_preprocessor()),
        # Pembobotan mengikuti keputusan Bagian 4a, supaya pohon ilustrasi ini
        # tidak memakai perlakuan yang berbeda dari model yang dilaporkan.
        ("clf", DecisionTreeClassifier(
            max_depth=3, random_state=RANDOM_STATE,
            class_weight=("balanced" if PENYEIMBANG_DIPILIH[nama] == "class_weight"
                          else None))),
    ]).fit(d["X_train"][FEATURES], d["y_train"])
    pohon_sederhana[nama] = pohon

    plt.figure(figsize=(24, 11))
    plot_tree(
        pohon.named_steps["clf"],
        feature_names=pohon.named_steps["pre"].get_feature_names_out(),
        class_names=["Tidak Berisiko", "Berisiko"],
        filled=True, rounded=True, fontsize=11, proportion=True,
    )
    plt.title(f"Decision Tree (max_depth=3) — {nama}   |   usia {d['usia']}", fontsize=16)
    plt.tight_layout()
    plt.show()

In [ ]:

# ===================== FITUR PEMISAH TERATAS =====================
# Fitur apa yang dipilih tiap pohon pada split paling atas?
baris = []
for nama, pohon in pohon_sederhana.items():
    clf = pohon.named_steps["clf"]
    fitur = pohon.named_steps["pre"].get_feature_names_out()
    baris.append({
        "Dataset": nama,
        "Fitur split teratas": fitur[clf.tree_.feature[0]],
        "Ambang (skala standar)": round(float(clf.tree_.threshold[0]), 3),
        "Kedalaman pohon penuh (hasil tuning)":
            params[(nama, "decision_tree")].get("clf__max_depth", "-"),
    })

display(pd.DataFrame(baris))


### Yang perlu diperhatikan pada pohon Diabetes

Split paling atas pada Diabetes memisahkan **`IMT_recalculated`**, dan simpul di bawahnya
hampir murni — satu sisi hampir seluruhnya "Berisiko", sisi lain hampir seluruhnya "Tidak
Berisiko".

Ini sesuai dengan temuan di Bagian 2: label diabetes pada dasarnya mengikuti aturan
**`IMT > 25`** dari kuesioner skrining. Artinya model tidak sedang mempelajari pola klinis
yang rumit — ia **membaca ulang aturan penilaian kuesioner**.

Konsekuensinya untuk laporan: **akurasi tinggi pada Diabetes tidak boleh disajikan sebagai
keberhasilan prediksi penyakit.** Bukti lengkapnya ada di `Analisis Data Mentah.ipynb`
Bagian 8 dan `Klasifikasi BMI.ipynb` Bagian 7.

Bandingkan dengan pohon TBC dan Kanker Paru: split teratasnya bukan IMT, dan simpulnya jauh
lebih bercampur — tanda bahwa polanya memang lebih sulit dan tidak berasal dari aturan
sederhana.


---
# 7. Feature Importance

Seberapa besar tiap fitur dipakai model XGBoost untuk mengambil keputusan. Nilainya berasal
dari `feature_importances_` bawaan (jenis **gain**): seberapa besar perbaikan yang diberikan
sebuah fitur setiap kali dipakai memisah data. Angkanya dinormalisasi sehingga totalnya 1.

Bagian **7b** kemudian menghitung ulang pertanyaan yang sama dengan **SHAP**, dan pada satu
penyakit jawabannya ternyata berbeda.

## Kenapa nilainya perlu dikelompokkan dulu

Setelah One-Hot Encoding, satu kolom asli bisa pecah menjadi banyak kolom. `segmen` yang
punya 7 kategori menjadi **6 kolom terpisah**, dan XGBoost melaporkan importance untuk tiap
kolom itu sendiri-sendiri.

Akibatnya kolom yang terpecah **terlihat jauh lebih kecil daripada pengaruh sebenarnya**.
Pada TBC misalnya, tidak ada satu pun kolom `segmen_*` yang tampak menonjol, padahal kalau
keenamnya dijumlahkan:

| TBC | Sebelum dikelompokkan | Setelah dikelompokkan |
|---|---|---|
| Fitur teratas | `Usia` (0.301) | **`segmen` (0.494)** |
| Peringkat kedua | `segmen_APBN` (0.215) | `Usia` (0.301) |

Kesimpulannya **berbalik**: yang paling berpengaruh pada TBC ternyata segmen kepesertaan,
bukan usia. Karena itu grafik utama di bawah memakai versi yang sudah dikelompokkan, dan
rincian per kolom tetap ditampilkan sebagai tabel di bawahnya.

*(Angka pada tabel di atas berasal dari run sebelumnya dan bisa bergeser sedikit setelah
notebook dijalankan ulang; yang tidak berubah adalah polanya — pengelompokan menaikkan
`segmen` ke peringkat teratas.)*


In [ ]:

# ===================== KELOMPOKKAN KOLOM HASIL ENCODING =====================
def kelompok_fitur(nama_kolom):
    # "num__Usia"            -> "Usia"
    # "cat__segmen_BP PN"    -> "segmen"
    # "cat__Jeniskelamin_P"  -> "Jeniskelamin"
    if nama_kolom.startswith("num__"):
        return nama_kolom[len("num__"):]
    sisa = nama_kolom[len("cat__"):]
    for c in CATEG_COLS:
        if sisa.startswith(c + "_"):
            return c
    return sisa


# Tunjukkan pemetaannya supaya jelas kolom mana masuk ke kelompok mana.
display(pd.DataFrame({
    "Kolom setelah encoding": NAMA_FITUR,
    "Fitur asli": [kelompok_fitur(f) for f in NAMA_FITUR],
}))

In [ ]:

# ===================== FEATURE IMPORTANCE (DIKELOMPOKKAN) =====================
importance_rinci, importance_grup = {}, {}

for nama in data:
    clf = models[(nama, "xgboost")].named_steps["clf"]
    imp = pd.Series(clf.feature_importances_, index=NAMA_FITUR)
    importance_rinci[nama] = imp
    # Importance kolom-kolom dari satu fitur asli dijumlahkan.
    importance_grup[nama] = imp.groupby(kelompok_fitur).sum()

grup_df = pd.DataFrame(importance_grup)

fig, axes = plt.subplots(2, 2, figsize=vz.FIG_2X2)

for ax, (nama, d) in zip(axes.flat, data.items()):
    s = grup_df[nama].sort_values()
    ax.barh(s.index, s.values, color="steelblue")
    ax.set_title(f"{nama} (usia {d['usia']})")
    ax.set_xlabel("Importance (gain, dijumlahkan per fitur asli)")
    for i, v in enumerate(s.values):
        ax.text(v, i, f" {v:.3f}", va="center", fontsize=10)
    ax.margins(x=0.15)

fig.suptitle("Feature Importance XGBoost — dikelompokkan per fitur asli", y=1.00)
plt.tight_layout()
plt.show()

print("Importance per fitur asli (total tiap kolom = 1.0):")
display(grup_df.round(4).sort_values("TBC", ascending=False))

In [ ]:

# ===================== RINCIAN PER KOLOM HASIL ENCODING =====================
# Berguna untuk melihat kategori mana di dalam 'segmen' yang paling berpengaruh.
rinci_df = pd.DataFrame(importance_rinci)
rinci_df.insert(0, "Fitur asli", [kelompok_fitur(f) for f in rinci_df.index])
display(rinci_df.round(4).sort_values(["Fitur asli"]))


Setelah dikelompokkan, polanya jauh lebih jelas dan konsisten dengan analisis sebelumnya:

- **Kanker Paru** — didominasi **`Jeniskelamin`**. Dari analisis data bersih, prevalensinya
  19.9% pada laki-laki melawan 2.7% pada perempuan. Jenis kelamin di sini berfungsi sebagai
  **proksi kebiasaan merokok**, karena status merokok sendiri tidak tersedia di data.
- **Diabetes & Hipertensi** — didominasi **`IMT_recalculated`**, sesuai aturan ambang IMT
  pada kuesioner.
- **TBC** — didominasi **`segmen`**, disusul `Usia`. Karena `segmen` adalah kategori
  kepesertaan BPJS, ini merupakan **proksi kondisi sosial ekonomi** — sejalan dengan
  literatur bahwa risiko TBC berkaitan erat dengan faktor sosial ekonomi.

**Catatan kehati-hatian.** `feature_importances_` menunjukkan fitur mana yang **paling sering
dan paling berguna dipakai model untuk memisah data**, bukan hubungan sebab-akibat. Segmen
kepesertaan tidak menyebabkan TBC; ia menandai kelompok yang risikonya memang lebih tinggi.

Peringatan kedua, yang justru lebih penting: *gain* mengukur seberapa berguna sebuah fitur
**di dalam struktur pohon**, bukan seberapa besar pengaruhnya **terhadap prediksi akhir**.
Keduanya tidak selalu sejalan. Bagian 7b memeriksanya.



---
# 7b. SHAP — Interpretabilitas Model

`feature_importances_` di Bagian 7 hanya menjawab satu pertanyaan: seberapa berguna sebuah
fitur ketika dipakai memisah data di dalam pohon. Ia tidak menjawab pertanyaan yang
sebenarnya ingin diketahui pemangku kepentingan: **untuk peserta ini, apa yang mendorong
prediksinya naik atau turun, dan sebesar apa?**

**SHAP** (*SHapley Additive exPlanations*, Lundberg & Lee, 2017) menjawab pertanyaan itu.
Ia meminjam konsep *Shapley value* dari teori permainan: tiap fitur diperlakukan sebagai
"pemain" yang ikut menyumbang pada hasil akhir, dan nilai SHAP adalah rata-rata kontribusi
satu fitur terhadap prediksi, dihitung atas semua urutan kemungkinan masuknya fitur.

Dua sifat yang membuatnya berguna di sini:

- **Aditif.** Untuk satu peserta, jumlah seluruh nilai SHAP ditambah nilai dasar
  (*base value*, yaitu prediksi rata-rata populasi) sama persis dengan prediksi model untuk
  peserta itu. Penjelasannya tidak bisa "tidak menjumlah".
- **Bertanda.** Nilai positif mendorong ke arah "Berisiko", nilai negatif menjauhinya —
  sesuatu yang `feature_importances_` sama sekali tidak bisa katakan, karena nilainya selalu
  positif.

Dua tingkat penjelasan dihitung di bawah:

| Tingkat | Pertanyaan | Grafik |
|---|---|---|
| **Global** | Fitur apa yang paling berpengaruh pada model secara keseluruhan? | bar (dikelompokkan) + beeswarm |
| **Lokal** | Kenapa peserta *ini* diprediksi berisiko? | waterfall |

> ⚠️ **Pengelompokan tetap wajib, persis seperti di Bagian 7.** SHAP juga dihitung per kolom
> hasil encoding, jadi `segmen` kembali terpecah menjadi 6 potong. Nilai SHAP dijumlahkan
> **dalam bentuk mutlak** (`mean|SHAP|`) per fitur asli sebelum diplot — kalau tidak, `segmen`
> akan tampak kecil karena alasan yang sama seperti sebelumnya.


In [ ]:

# ===================== SHAP: HITUNG UNTUK KEEMPAT PENYAKIT =====================
# Seperti Bagian 7, SHAP selalu dihitung pada xgboost untuk keempat penyakit,
# supaya perbandingan antar penyakit setara.
N_SHAP = 20_000        # subsampel data uji; TreeExplainer sendiri cepat, ini demi grafik

shap_nilai, shap_data, shap_grup = {}, {}, {}

for nama, d in data.items():
    pipe = models[(nama, "xgboost")]
    pre, clf = pipe.named_steps["pre"], pipe.named_steps["clf"]

    X_enc = pd.DataFrame(pre.transform(d["X_test"][FEATURES]), columns=NAMA_FITUR)
    X_enc = X_enc.head(N_SHAP)

    sv = shap.TreeExplainer(clf)(X_enc)
    # Sebagian model (mis. DecisionTree) mengembalikan (n, fitur, kelas);
    # ambil kelas positif supaya bentuknya selalu (n, fitur).
    if sv.values.ndim == 3:
        sv = sv[:, :, 1]
    nilai = sv.values

    shap_nilai[nama], shap_data[nama] = sv, X_enc

    # Dikelompokkan kembali ke fitur asli, memakai fungsi yang sama dengan Bagian 7.
    g = pd.Series(np.abs(nilai).mean(axis=0), index=NAMA_FITUR).groupby(kelompok_fitur).sum()
    shap_grup[nama] = g / g.sum()

    print(f"{nama:20} SHAP dihitung pada {len(X_enc):,} baris uji")


In [ ]:

# ===================== SHAP GLOBAL vs GAIN — BERDAMPINGAN =====================
fig, axes = plt.subplots(2, 2, figsize=vz.FIG_2X2)

for ax, (nama, d) in zip(axes.flat, data.items()):
    banding = pd.DataFrame({
        "gain": grup_df[nama],
        "SHAP": shap_grup[nama],
    }).sort_values("SHAP")

    y = np.arange(len(banding))
    ax.barh(y - 0.2, banding["gain"], height=0.4, color="steelblue", label="gain")
    ax.barh(y + 0.2, banding["SHAP"], height=0.4, color="darkorange", label="mean|SHAP|")
    ax.set_yticks(y)
    ax.set_yticklabels(banding.index)
    ax.set_xlabel("Importance (dinormalisasi, dijumlahkan per fitur asli)")
    ax.set_title(f"{nama} (usia {d['usia']})")
    ax.legend(fontsize=9, loc="lower right")
    ax.margins(x=0.15)

fig.suptitle("Feature Importance: gain (Bagian 7) vs mean|SHAP| (Bagian 7b)", y=1.00)
plt.tight_layout()
plt.show()

# Apakah keduanya sepakat soal fitur teratas?
banding_puncak = pd.DataFrame({
    "Teratas menurut gain": {n: grup_df[n].idxmax() for n in data},
    "Teratas menurut SHAP": {n: shap_grup[n].idxmax() for n in data},
})
banding_puncak["Sepakat?"] = np.where(
    banding_puncak["Teratas menurut gain"] == banding_puncak["Teratas menurut SHAP"],
    "ya", "TIDAK")
display(banding_puncak)


In [ ]:

# ===================== SHAP GLOBAL: BEESWARM PER PENYAKIT =====================
# Beeswarm menampilkan satu titik per peserta: posisi mendatar = besar kontribusinya,
# warna = tinggi/rendahnya nilai fitur itu pada peserta tersebut.
for nama, d in data.items():
    shap.plots.beeswarm(shap_nilai[nama], max_display=len(NAMA_FITUR), show=False)
    plt.title(f"SHAP — {nama} (usia {d['usia']}), model xgboost", fontsize=13)
    plt.tight_layout()
    plt.show()


In [ ]:

# ===================== SHAP LOKAL: KENAPA SATU PESERTA DIPREDIKSI BERISIKO =====================
# Diambil satu peserta TBC yang benar-benar berlabel Berisiko dan ditandai model.
nama = "TBC"
y_te = data[nama]["y_test"].to_numpy()[:N_SHAP]
p_te = probs[(nama, "xgboost")][:N_SHAP]
kandidat = np.where((y_te == 1) & (p_te >= 0.5))[0]
if len(kandidat) == 0:                      # tidak ada positif yang tertangkap
    kandidat = np.argsort(-p_te)[:1]        # ambil yang probabilitasnya tertinggi
i = int(kandidat[0])

asli = data[nama]["X_test"][FEATURES].iloc[i]
print(f"Peserta ke-{i} pada data uji {nama}:")
for k, v in asli.items():
    print(f"   {k:20} {v}")
print(f"   {'probabilitas model':20} {p_te[i]:.4f}   (label sebenarnya: Berisiko)")

shap.plots.waterfall(shap_nilai[nama][i], max_display=len(NAMA_FITUR), show=False)
plt.title(f"SHAP lokal — satu peserta {nama}", fontsize=13)
plt.tight_layout()
plt.show()

# Sifat aditif SHAP: base value + seluruh kontribusi = keluaran model (skala log-odds).
sv = shap_nilai[nama]
total = float(sv.values[i].sum() + sv.base_values[i])
margin = float(models[(nama, "xgboost")].named_steps["clf"].predict(
    shap_data[nama].iloc[[i]].to_numpy(), output_margin=True)[0])
print(f"\nbase value + jumlah SHAP = {total:.6f}")
print(f"keluaran model (log-odds) = {margin:.6f}")
print(f"selisih                   = {abs(total - margin):.2e}  -> aditif, seperti seharusnya")



### Yang ditemukan SHAP

**1. Pada tiga penyakit, SHAP dan gain sepakat.** Diabetes dan Hipertensi tetap didominasi
`IMT_recalculated`, Kanker Paru tetap didominasi `Jeniskelamin`. Kesimpulan Bagian 7 tidak
berubah untuk ketiganya.

**2. Pada TBC keduanya tidak sepakat, dan ini temuan tersendiri.** Gain menempatkan `segmen`
di peringkat teratas, sedangkan mean|SHAP| menempatkan **`Usia`** jauh di atas segalanya,
dengan `segmen` turun ke peringkat kedua.

Keduanya tidak saling membatalkan, karena keduanya mengukur hal yang berbeda:

| | Yang diukur | Kenapa `segmen` besar/kecil di sini |
|---|---|---|
| **gain** | total perbaikan pemisahan setiap kali sebuah fitur dipakai memisah, dijumlahkan atas seluruh pohon | `segmen` terpecah jadi 6 kolom dan dipakai di **banyak** simpul kecil; tiap pemakaian menyumbang gain, dan jumlahnya jadi besar |
| **mean\|SHAP\|** | rata-rata besar pengaruh sebuah fitur terhadap **prediksi akhir**, dihitung pada peserta nyata | banyak simpul `segmen` hanya mengenai sedikit peserta, jadi pengaruh rata-ratanya terhadap prediksi kecil |

**Pembacaan yang mana yang dipakai?** Untuk pertanyaan "apa yang menggerakkan prediksi
model pada peserta sungguhan", SHAP yang lebih tepat — dan jawabannya, **usia**. Pembacaan
ini juga sejalan dengan analisis yang sama sekali tidak melibatkan model: pada tabel filter
di Bagian 3a, hubungan terkuat dengan TBC adalah `Usia` (0.125), di atas `segmen` (0.099).
Dua metode yang berdiri sendiri menunjuk ke arah yang sama.

Ini sekaligus **memperjelas, bukan membatalkan**, temuan bahwa segmen kepesertaan penting
bagi TBC. `segmen` tetap berpengaruh nyata dan tetap bertahan setelah usia dikendalikan —
lihat `Analisis Hubungan Antar Variabel.ipynb`, di mana Cramér's V segmen–TBC naik dari
0.030 pada usia 15–29 menjadi 0.098 pada 60+. Yang dikoreksi SHAP hanyalah **besarnya**:
sebagian dari angka gain `segmen` yang besar itu sebenarnya efek usia yang menumpang, bukan
sinyal sosial ekonomi murni.

**3. Penjelasan lokal terbukti aditif.** Selisih antara `base value + Σ SHAP` dan keluaran
model berada di orde 1e-07 — yaitu nol dalam batas presisi mesin. Inilah sifat yang membuat
SHAP layak ditampilkan kepada peserta: angkanya benar-benar menjumlah menjadi prediksinya
sendiri, bukan sekadar ilustrasi.

Penjelasan lokal inilah yang kemudian ditampilkan per peserta di dasbor
(`Dashboard/app_streamlit.py`).



---
# 8. Kesimpulan

In [ ]:

# ===================== TABEL GABUNGAN =====================
gabungan = pd.concat(
    [tab.assign(Dataset=nama) for nama, tab in tabel_per_penyakit.items()],
    ignore_index=True,
)
gabungan["Rentang usia"] = gabungan["Dataset"].map({k: v[2] for k, v in DISEASES.items()})
gabungan = gabungan[["Dataset", "Rentang usia", "Prevalensi", "Model",
                     "Accuracy", "Precision", "Recall", "F1-Score", "ROC-AUC"]]

display(gabungan.set_index(["Dataset", "Model"]).round(4))

out = os.path.join(DATA_DIR, "ringkasan_model.csv")
gabungan.to_csv(out, index=False, encoding="utf-8-sig")
print(f"Tersimpan: {out}")

In [ ]:

# ===================== EKSPOR HYPERPARAMETER TERPILIH =====================
# Ditulis ke berkas supaya Dashboard/train_export.py TIDAK perlu menyalin
# angka-angka ini dengan tangan. Penyalinan manual adalah sumber kesalahan
# yang tidak terlihat: nilai yang salah ketik tetap menghasilkan model yang
# jalan, hanya metriknya yang diam-diam berbeda dari yang dilaporkan.
import json as _json
from datetime import datetime as _dt

_hp = {
    "dibuat": _dt.now().isoformat(timespec="seconds"),
    "wilayah": WILAYAH,
    "random_state": RANDOM_STATE,
    "quick_mode": QUICK_MODE,
    "n_iter": N_ITER,
    "cv_folds": CV_FOLDS,
    "fitur": FEATURES,
    "penyakit": {},
}

for _nama, _tab in tabel_per_penyakit.items():
    _mk = model_terbaik(_tab.reset_index())
    _hp["penyakit"][_nama] = {
        "model_terbaik": _mk,
        "penyeimbang": PENYEIMBANG_DIPILIH[_nama],
        "pita_usia": DISEASES[_nama][2],
        "prevalensi": float(data[_nama]["base_rate"]),
        "params": {k.replace("clf__", ""): v
                   for k, v in params[(_nama, _mk)].items()},
    }

_out_hp = os.path.join(DATA_DIR, "hyperparameter_terpilih.json")
with open(_out_hp, "w", encoding="utf-8") as _f:
    _json.dump(_hp, _f, indent=1, ensure_ascii=False)

print(f"Tersimpan: {_out_hp}")
for _nama, _v in _hp["penyakit"].items():
    print(f"   {_nama:<18} {_v['model_terbaik']:<15} "
          f"penyeimbang={_v['penyeimbang']}  {_v['params']}")

In [ ]:

# ===================== MODEL TERBAIK PER PENYAKIT =====================
terbaik = []
for nama, tab in tabel_per_penyakit.items():
    t = tab[tab["Model"] != "dummy"]
    r = t.loc[t["F1-Score"].idxmax()]
    dm = tab[tab["Model"] == "dummy"].iloc[0]
    terbaik.append({
        "Dataset": nama,
        "Rentang usia": data[nama]["usia"],
        "Prevalensi (%)": round(data[nama]["base_rate"] * 100, 2),
        "Ketimpangan": NAMA_OPSI[PENYEIMBANG_DIPILIH[nama]],
        "Model terbaik": r["Model"],
        "Accuracy": round(r["Accuracy"], 4),
        "Precision": round(r["Precision"], 4),
        "Recall": round(r["Recall"], 4),
        "F1-Score": round(r["F1-Score"], 4),
        "ROC-AUC": round(r["ROC-AUC"], 4),
        "F1 dummy": round(dm["F1-Score"], 4),
    })

display(pd.DataFrame(terbaik))

if QUICK_MODE:
    print()
    print("=" * 72)
    print("PERINGATAN: angka di atas dihasilkan dengan QUICK_MODE = True.")
    print("Setel QUICK_MODE = False di Bagian 1, jalankan ulang seluruh notebook,")
    print("lalu pakai hasil yang baru untuk laporan akhir.")
    print("=" * 72)



## Ringkasan temuan

**1. Setiap model hanya berlaku pada rentang usianya.**

| Model | Berlaku untuk |
|---|---|
| Diabetes Melitus | usia **15+** |
| Hipertensi | usia **15+** |
| Kanker Paru | usia **45+** |
| TBC | **semua usia** |

Label `Tidak Dapat Pertanyaan` pada data asli berarti responden **tidak memenuhi syarat
usia** untuk ditanyai, bukan data hilang. Baris tersebut sudah dibuang di tahap persiapan
data. Rentang ini wajib disebutkan setiap kali hasil dilaporkan.

**2. Hasil Diabetes tidak boleh dibaca sebagai keberhasilan prediksi.**

Visualisasi pohon di Bagian 6 memperlihatkan model memisahkan data terutama berdasarkan IMT
di sekitar nilai 25 — sama dengan aturan penilaian pada kuesioner skrining. Model
merekonstruksi aturan tersebut, bukan memprediksi penyakit. SHAP di Bagian 7b menunjukkan
hal yang sama dari sudut lain: kontribusi `IMT_recalculated` berubah tajam tepat di satu
titik, bukan menanjak perlahan seperti hubungan biologis pada umumnya.

**3. Kanker Paru dan TBC memang lebih sulit, dan itu bukan kesalahan model.**

Fitur yang tersedia hanya demografi dan antropometri. Kanker Paru hampir seluruhnya
ditentukan jenis kelamin (proksi merokok), TBC oleh usia dan segmen. Faktor risiko yang
paling menentukan justru **tidak ada di data**.

**4. Ketimpangan kelas wajib ditangani, tetapi cara mana yang dipakai hampir tidak
berpengaruh.**

Bagian 4a membandingkan empat pendekatan pada Kanker Paru dan TBC. Tanpa penanganan, recall
runtuh ke nol meski akurasi tertinggi. Dengan penanganan — class weight, SMOTE, atau
undersampling — hasilnya berbeda kurang dari 0.01 F1. Yang menentukan adalah **memakai
salah satunya**, bukan memilih yang mana.

Temuan sampingan yang layak dilaporkan: **ROC-AUC nyaris tidak bereaksi** terhadap
penyeimbangan kelas, karena penyeimbangan menggeser ambang tanpa mengubah urutan peringkat.

**5. Perbandingan model.**

Model terbaik per penyakit dipilih menurut F1-Score dari tiga kandidat sungguhan: regresi
logistik, pohon keputusan, dan XGBoost. `dummy` selalu dikecualikan — ia garis dasar, bukan
kandidat. Perlu diingat bahwa selisih antar model yang kecil (di bawah ~0.01 F1) tidak cukup
untuk menyimpulkan satu model lebih baik daripada yang lain — bandingkan juga ROC-AUC-nya
sebelum menyimpulkan.

**6. Gain dan SHAP bisa memberi jawaban berbeda, dan pada TBC memang berbeda.**

Bagian 7 (gain) menempatkan `segmen` teratas untuk TBC; Bagian 7b (SHAP) menempatkan `Usia`.
Keduanya benar untuk pertanyaan masing-masing. Untuk pertanyaan "apa yang menggerakkan
prediksi pada peserta nyata", SHAP yang dipakai.

## Catatan pembacaan

- **Bagian 5 (confusion matrix)** memakai *model terbaik per penyakit*, lewat fungsi
  `model_terbaik()` yang sama dengan tabel di atasnya — jadi keduanya tidak mungkin
  menunjuk model yang berbeda.
- **Bagian 7 dan 7b (feature importance & SHAP)** selalu memakai `xgboost` untuk keempat
  penyakit, supaya perbandingan antar penyakit setara. Jadi bila model terbaik suatu
  penyakit bukan xgboost, kedua bagian ini memakai model yang berbeda dari Bagian 5 — ini
  disengaja, bukan ketidakkonsistenan.
- **`dummy` bukan kandidat.** Ia dilatih hanya sebagai garis dasar pembanding dan
  dikecualikan dari pemilihan model terbaik oleh `model_terbaik()`.

## Keterbatasan

- **Variabel penting tidak tersedia**: riwayat keluarga, lingkar perut, status merokok,
  tekanan darah, dan aktivitas fisik. Inilah penyebab utama batas atas performa, bukan
  pilihan algoritma.
- **Kolom `id` teracak**, sehingga tidak mungkin memastikan satu orang hanya terhitung
  sekali. Semua angka berlaku per catatan skrining.
- **Label Diabetes bersifat turunan aturan**, sehingga tidak layak dipakai sebagai acuan
  klinis.
- Populasi adalah **peserta yang ikut skrining**, bukan seluruh penduduk Sleman.
- **Panel prevalensi menurut kelompok IMT untuk TBC masih terpengaruh usia.** TBC satu-satunya
  dataset yang mencakup usia 0–106, dan anak-anak punya IMT rendah sekaligus prevalensi TBC
  rendah. Kenaikan prevalensi terhadap IMT pada panel itu belum tentu efek IMT.
- **Perbandingan di Bagian 4a memakai hyperparameter tetap**, bukan hasil penyetelan per
  pendekatan. Kesimpulannya ("ketiga penanganan setara") berlaku pada kondisi itu; penyetelan
  penuh per pendekatan bisa saja menggeser selisihnya sedikit, meski tidak mungkin mengubah
  fakta bahwa "tanpa penanganan" runtuh recall-nya.
